# **Testing REST API**

## **Import Library**

In [1]:
import json
import requests
import base64
import tensorflow as tf

# URL endpoint
SERVED_MODEL_URL = "http://localhost:8501/v1/models/heart-failure-model"

2026-08-15 06:41:50.339420: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-15 06:41:50.373291: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-15 06:41:50.374414: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-08-15 06:41:51.283548: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## **Uji Status & Metadata Model**

In [2]:
# Mengecek metadata signature
metadata_url = f"{SERVED_MODEL_URL}/metadata"
response = requests.get(metadata_url)

print("Status Code:", response.status_code)
print("\nModel Metadata Response:")
print(json.dumps(response.json(), indent=2))

Status Code: 200

Model Metadata Response:
{
  "model_spec": {
    "name": "heart-failure-model",
    "version": "1786775813",
    "signature_name": ""
  },
  "metadata": {
    "signature_def": {
      "signature_def": {
        "__saved_model_init_op": {
          "inputs": {},
          "outputs": {
            "__saved_model_init_op": {
              "name": "NoOp",
              "dtype": "DT_INVALID",
              "tensor_shape": {
                "dim": [],
                "unknown_rank": true
              }
            }
          },
          "method_name": "",
          "defaults": {}
        },
        "serving_default": {
          "inputs": {
            "examples": {
              "name": "serving_default_examples:0",
              "dtype": "DT_STRING",
              "tensor_shape": {
                "dim": [
                  {
                    "size": "-1",
                    "name": ""
                  }
                ],
                "unknown_rank": false
   

Tahap ini bertujuan untuk memverifikasi status operasional server dan struktur interface model sebelum melakukan inferensi:

- **Pengecekan Status:** Memastikan container TensorFlow Serving berjalan aktif dan dapat merespons HTTP Request dengan baik (ditandai dengan Status Code `200 OK`).
- **Inspeksi Signature:** Memeriksa nama signature (`serving_default`), tipe data input (`DT_STRING`), serta bentuk tensor output yang siap menerima permintaan prediksi.

## **Uji Prediksi Data Sampel**

In [3]:
# Helper untuk mengonversi data
def create_tf_example(data):
    feature = {}
    for key, val in data.items():
        if isinstance(val, float):
            feature[key] = tf.train.Feature(float_list=tf.train.FloatList(value=[val]))
        elif isinstance(val, int):
            feature[key] = tf.train.Feature(int64_list=tf.train.Int64List(value=[val]))
        elif isinstance(val, str):
            feature[key] = tf.train.Feature(
                bytes_list=tf.train.BytesList(value=[val.encode("utf-8")])
            )

    example_proto = tf.train.Example(features=tf.train.Features(feature=feature))
    return example_proto.SerializeToString()


# Data tes pasien
sample_patient = {
    "Age": 40,
    "RestingBP": 140,
    "Cholesterol": 289,
    "FastingBS": 0,
    "MaxHR": 172,
    "Oldpeak": 0.0,
    "Sex": "M",
    "ChestPainType": "ATA",
    "RestingECG": "Normal",
    "ExerciseAngina": "N",
    "ST_Slope": "Up",
}

# Serialisasi data ke format b64 string
serialized_example = create_tf_example(sample_patient)
b64_example = base64.b64encode(serialized_example).decode("utf-8")

# Payload request prediksi
payload = {"signature_name": "serving_default", "instances": [{"b64": b64_example}]}

# POST request
predict_url = f"{SERVED_MODEL_URL}:predict"
response = requests.post(predict_url, json=payload)
result = response.json()

print("Prediction Endpoint Status Code:", response.status_code)
print("Response Result:", result)

# Interpretasi Hasil
prediction_score = result["predictions"][0][0]
print(f"\nSkor Probabilitas Penyakit Jantung: {prediction_score:.4f}")
if prediction_score >= 0.5:
    print("Hasil Diagnosa: Berisiko Tinggi Penyakit Jantung (1)")
else:
    print("Hasil Diagnosa: Normal / Risiko Rendah (0)")

Prediction Endpoint Status Code: 200
Response Result: {'predictions': [[0.11237926]]}

Skor Probabilitas Penyakit Jantung: 0.1124
Hasil Diagnosa: Normal / Risiko Rendah (0)


Tahap ini mensimulasikan proses inferensi data pasien baru secara real-time via REST API dengan alur kerja sebagai berikut:

- **Serialisasi Data Input:** Fungsi `create_tf_example` mengonversi dictionary data pasien menjadi protobuf `tf.train.Example`, diserialisasi, lalu di-encode ke format Base64 agar sesuai dengan spesifikasi input SavedModel.
- **Pengiriman Request:** Mengirimkan HTTP **POST Request** yang membawa payload Base64 ke endpoint `:predict`.
- **Interpretasi Hasil:** Menerima respons JSON berupa skor probabilitas (rentang 0–1) dan mengategorikannya ke dalam keputusan klinis sederhana (skor `>= 0.5` diklasifikasikan sebagai **Berisiko Tinggi**).